In [ ]:
# Upload the ZIP created by rag_audio_retrieval.py.
from google.colab import files
from pathlib import Path
import io
import zipfile
import json

uploaded = files.upload()
OUTPUT_DIR = Path("audio_rag")
with zipfile.ZipFile(io.BytesIO(uploaded[next(iter(uploaded))])) as archive:
    archive.extractall(OUTPUT_DIR)

rag_context = json.loads((OUTPUT_DIR / "rag_context.json").read_text(encoding="utf-8"))
print("Input audio:", rag_context["query"])
print("Question:", rag_context["question"])
for modality, items in rag_context["retrieved"].items():
    print(f"\n{modality.upper()} RESULTS:")
    for item in items:
        print("Species:", item["species"])
        print("Distance:", round(item["distance"], 4))
        if modality == "text":
            print("Description:", (item.get("description") or "")[:200])


In [ ]:
# Select the closest retrieved image as a reference.
from PIL import Image
from IPython.display import display

retrieved = rag_context["retrieved"]
best_image = min(retrieved["image"], key=lambda item: item["distance"])
image_path = OUTPUT_DIR / best_image["image_file"]
image = Image.open(image_path).convert("RGB")
print("Retrieved reference image:", best_image["species"])
display(image)


In [ ]:
# Build the context from the three retrieved modalities.
question = rag_context["question"]
query = rag_context["query"]
text_context = "\n\n".join(
    f"Species: {item['species']}\nDescription: {item.get('description') or ''}"
    for item in retrieved["text"]
)
image_context = "\n".join(f"Image {i+1}: {item['species']}" for i, item in enumerate(retrieved["image"]))
audio_context = "\n".join(f"Audio {i+1}: {item['species']}" for i, item in enumerate(retrieved["audio_projected"]))
prompt = f"""You are an expert ornithologist.
Question: {question}

An external bird recording was used to retrieve evidence from a multimodal database.
Retrieved text descriptions:
{text_context}

Retrieved image metadata:
{image_context}

Retrieved audio species metadata:
{audio_context}

The provided image is the closest retrieved reference image, labelled {best_image['species']}.
It is not a photograph of the bird in the input recording.
Identify the best supported candidate for the bird in the recording using the retrieved evidence.
Use the image as supporting evidence; if the modalities disagree, state uncertainty.
You have not heard the recording. Do not claim to have listened to or analysed its sound.
Do not infer the species from a filename. Do not invent unsupported details.
Answer briefly.
"""
print(prompt)


In [ ]:
%pip install -q transformers accelerate bitsandbytes


In [ ]:

import torch
from transformers import (
    LlavaForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

# Load LLaVA 
model_id = "llava-hf/llava-1.5-7b-hf"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)

print("LLaVA loaded successfully!")


In [ ]:
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }
]

formatted_prompt = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True
)

inputs = processor(
    images=image,
    text=formatted_prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

answer = processor.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("\nUSER QUERY:", query)
print("SELECTED IMAGE:", image_path)
print("\nLLaVA ANSWER:")
print(answer)
